# Build a validated regression set

Python 3.12 · offline · estimated time: 18 minutes

You will write the regression criterion and complete ten labeled records. The notebook supplies validation and persistence for immediate feedback.

Replace every `TODO`, then run each cell in order. The final cell persists `eval_cases.jsonl`, which the next lesson will consume.

In [2]:
# TODO 1: State one observable regression using the input and candidate output.
criterion = ''

# Check 1: Confirm the criterion is present before defining the label meanings.
assert criterion.strip(), 'TODO 1: write the regression criterion'
label_meaning = {1: 'regression present', 0: 'acceptable under this criterion'}
print('CHECK 1 — criterion:', criterion)
print('CHECK 1 — label mapping: 1 = regression present; 0 = acceptable')

AssertionError: TODO 1: write the regression criterion

In [ ]:
# TODO 2: Write each candidate reply, then label it against your criterion.
# Keep the IDs, inputs, and four-field interface unchanged.
records = [
    {
        'case_id': 'refund-window-001',
        'input': 'Policy: refunds are available within 30 days. Request: day 45.',
        'candidate_output': '',  # Write an incorrect reply that grants a refund.
        'human_label': None,     # Replace with 0 or 1.
    },
    {
        'case_id': 'refund-window-002',
        'input': 'Policy: refunds are available within 30 days. Request: day 45.',
        'candidate_output': '',  # Write a correct reply that denies eligibility.
        'human_label': None,     # Replace with 0 or 1.
    },
    {
        'case_id': 'refund-window-003',
        'input': 'Policy: refunds are available within 30 days. Request: day 33.',
        'candidate_output': '',  # Write another incorrect reply that grants a refund.
        'human_label': None,     # Replace with 0 or 1.
    },
    {
        'case_id': 'refund-window-004',
        'input': 'Policy: refunds are available within 30 days. Request: day 12.',
        'candidate_output': '',  # Write a correct reply for an in-window request.
        'human_label': None,     # Replace with 0 or 1.
    },
    {
        'case_id': 'refund-window-005',
        'input': 'Policy: refunds are available within 30 days. Request: day 41.',
        'candidate_output': '',
        'human_label': None,
    },
    {
        'case_id': 'refund-window-006',
        'input': 'Policy: refunds are available within 30 days. Request: day 35.',
        'candidate_output': '',
        'human_label': None,
    },
    {
        'case_id': 'refund-window-007',
        'input': 'Policy: refunds are available within 30 days. Request: day 31.',
        'candidate_output': '',
        'human_label': None,
    },
    {
        'case_id': 'refund-window-008',
        'input': 'Policy: refunds are available within 30 days. Request: day 5.',
        'candidate_output': '',
        'human_label': None,
    },
    {
        'case_id': 'refund-window-009',
        'input': 'Policy: refunds are available within 30 days. Request: day 60.',
        'candidate_output': '',
        'human_label': None,
    },
    {
        'case_id': 'refund-window-010',
        'input': 'Policy: refunds are available within 30 days. Request: day 20.',
        'candidate_output': '',
        'human_label': None,
    },
]
# Check 2: Confirm the dataset is large enough and every record has a reply and binary label.
assert len(records) >= 10, 'add at least 10 cases for the cumulative grader'
assert all(row['candidate_output'].strip() for row in records), 'TODO 2: complete every candidate_output'
assert all(row['human_label'] in (0, 1) for row in records), 'TODO 2: label every case with 0 or 1'
print(f'CHECK 2 — created {len(records)} cases')
print('CHECK 2 — preview:', [(row['case_id'], row['human_label']) for row in records])


In [ ]:
from collections import Counter

REQUIRED = {'case_id': str, 'input': str, 'candidate_output': str, 'human_label': int}

def validate(rows):
    errors, seen = [], set()
    for number, row in enumerate(rows, start=1):
        for field, expected_type in REQUIRED.items():
            if field not in row:
                errors.append(f'row {number}: missing {field}')
            elif type(row[field]) is not expected_type:
                errors.append(f'row {number}: {field} must be {expected_type.__name__}')
            elif expected_type is str and not row[field].strip():
                errors.append(f'row {number}: {field} must not be empty')
        case_id = row.get('case_id')
        if isinstance(case_id, str) and case_id in seen:
            errors.append(f'row {number}: duplicate case_id {case_id!r}')
        seen.add(case_id)
        if type(row.get('human_label')) is int and row['human_label'] not in (0, 1):
            errors.append(f'row {number}: human_label must be 0 or 1')
    if errors:
        raise ValueError('\n'.join(errors))

# Check 3: Validate required fields, exact types, non-empty text, unique IDs, and binary labels.
validate(records)
print('CHECK 3 — schema, types, labels, and unique IDs: PASS')

In [ ]:
# Check 4: Count labels and require examples from both the regression and acceptable classes.
counts = Counter(row['human_label'] for row in records)
if not (counts[0] and counts[1]):
    raise ValueError('both classes are required: add at least one label 0 and one label 1')
print(f'CHECK 4 — class counts: label 0 = {counts[0]}, label 1 = {counts[1]}')
print('CHECK 4 — both-class course rule: PASS')

In [ ]:
import json
from pathlib import Path

artifact_path = (Path('build/lesson-01/eval_cases.jsonl') if Path('build/lesson-01').is_dir() else Path('eval_cases.jsonl'))
artifact_path.parent.mkdir(parents=True, exist_ok=True)
# Check 5: Save the JSONL artifact, reload it, and verify that no record changed.
artifact_path.write_text(''.join(json.dumps(row) + '\n' for row in records), encoding='utf-8')
reloaded = [json.loads(line) for line in artifact_path.read_text(encoding='utf-8').splitlines()]
assert reloaded == records, 'read-back records differ from in-memory records'
print(f'CHECK 5 — wrote and reloaded {len(reloaded)} records: PASS')
print('FINAL PASS — artifact ready for EP-02: build/lesson-01/eval_cases.jsonl')